Wersja N8N na dokerze wg. https://docs.n8n.io/hosting/installation/docker/#prerequisites


In [ ]:
docker volume create n8n_data

docker run -it --rm \
 --name n8n \
 -p 5678:5678 \
 -e GENERIC_TIMEZONE="<YOUR_TIMEZONE>" \
 -e TZ="<YOUR_TIMEZONE>" \
 -e N8N_ENFORCE_SETTINGS_FILE_PERMISSIONS=true \
 -e N8N_RUNNERS_ENABLED=true \
 -v n8n_data:/home/node/.n8n \
 docker.n8n.io/n8nio/n8n

Dla czasu Warszawa

In [ ]:
docker volume create n8n_data

docker run -it --rm \
  --name n8n \
  -p 5678:5678 \
  -e GENERIC_TIMEZONE="Europe/Warsaw" \
  -e TZ="Europe/Warsaw" \
  -e N8N_ENFORCE_SETTINGS_FILE_PERMISSIONS=true \
  -e N8N_RUNNERS_ENABLED=true \
  -v n8n_data:/home/node/.n8n \
  docker.n8n.io/n8nio/n8n

### Co robią te linie

docker volume create n8n_data tworzy trwały wolumen Dockera, dzięki któremu dane n8n nie znikną po zatrzymaniu kontenera.

W samym docker run:

--name n8n nadaje nazwę kontenerowi.

-p 5678:5678 wystawia aplikację lokalnie na porcie 5678.

-v n8n_data:/home/node/.n8n podłącza miejsce, w którym n8n trzyma konfigurację i dane.

N8N_ENFORCE_SETTINGS_FILE_PERMISSIONS=true i N8N_RUNNERS_ENABLED=true są zalecane w przykładzie dokumentacji n8n.

### Ważna uwaga praktyczna

Ten wariant używa --rm, więc po zatrzymaniu kontenera sam kontener zostanie usunięty, ale dane pozostaną, bo są zapisane w wolumenie n8n_data.

Jeśli chcesz, żeby n8n działał wygodniej w tle, zamiast w trybie interaktywnym terminala, zwykle lepiej użyć:

-d zamiast -it

usunąć --rm

Przykład bardziej praktyczny na co dzień:

In [ ]:
docker run -d \
  --name n8n \
  -p 5678:5678 \
  -e GENERIC_TIMEZONE="Europe/Warsaw" \
  -e TZ="Europe/Warsaw" \
  -e N8N_ENFORCE_SETTINGS_FILE_PERMISSIONS=true \
  -e N8N_RUNNERS_ENABLED=true \
  -v n8n_data:/home/node/.n8n \
  docker.n8n.io/n8nio/n8n

Dla Windows

Jeśli używasz PowerShell, zapis z backslash \ na końcu linii może nie działać tak jak w Linux/macOS, więc najprościej:

wkleić wszystko w jednej linii, albo

użyć kontynuacji linii zgodnej z PowerShell.

Wersja jednoliniowa do PowerShell:


In [ ]:
docker volume create n8n_data
docker run -d --name n8n -p 5678:5678 -e GENERIC_TIMEZONE="Europe/Warsaw" -e TZ="Europe/Warsaw" -e N8N_ENFORCE_SETTINGS_FILE_PERMISSIONS=true -e N8N_RUNNERS_ENABLED=true -v n8n_data:/home/node/.n8n docker.n8n.io/n8nio/n8n

Tak — w tej wersji przepływ danych jest lokalny, kontenerowy i trwały tylko dla katalogu .n8n. Najkrócej: przeglądarka łączy się z n8n na localhost:5678, n8n działa w kontenerze Dockera, a jego stan jest zapisywany w wolumenie n8n_data, więc po restarcie kontenera dane pozostają.

Jak płyną dane
Ty otwierasz panel w przeglądarce pod http://localhost:5678. Żądanie trafia do portu 5678 na komputerze gospodarzu, a Docker przekierowuje je do portu 5678 wewnątrz kontenera n8n.

Kontener n8n odbiera żądania HTTP i renderuje interfejs webowy oraz obsługuje API n8n. To jest warstwa aplikacyjna, czyli sama logika workflow, edycja, uruchamianie i zapis konfiguracji.

Dane trwałe są zapisywane do wolumenu n8n_data, zamapowanego na /home/node/.n8n. W tym katalogu n8n trzyma m.in. konfigurację instancji, klucze szyfrowania, ustawienia i lokalną bazę SQLite, jeśli nie ustawisz zewnętrznej bazy.

Wykonanie workflow dzieje się wewnątrz kontenera, a dzięki N8N_RUNNERS_ENABLED=true n8n korzysta z task runnerów, czyli oddzielnego mechanizmu wykonywania zadań zalecanego przez n8n.

Czas jest interpretowany zgodnie ze strefą Europe/Warsaw, ponieważ TZ ustawia strefę systemową kontenera, a GENERIC_TIMEZONE ustawia domyślną strefę n8n dla elementów czasowych, np. Schedule Trigger.

Co dzieje się technicznie w tym setupie
Ten układ składa się z trzech głównych „warstw”:

Host: Twój komputer z Dockerem.

Container: uruchomiona aplikacja n8n.

Volume: trwały magazyn danych n8n_data.

Przeglądarka nie zapisuje danych n8n lokalnie, tylko komunikuje się z kontenerem po HTTP. Jeśli zamkniesz przeglądarkę, n8n dalej działa, bo proces jest w kontenerze, a nie w samej sesji przeglądarki.

Jeśli zatrzymasz kontener, ale nie usuniesz wolumenu, to workflowy, credentiale i ustawienia nadal będą dostępne po ponownym uruchomieniu n8n.

Co jest trwałe, a co nietrwałe
Trwałe:

workflowy,

credentiale,

ustawienia instancji,

klucze i pliki zapisane w .n8n.

Nietrwałe:

sam kontener, jeśli go usuniesz i utworzysz od nowa,

pliki przechowywane poza wolumenem,

wszystko, co zapiszesz tylko „wewnątrz kontenera” poza /home/node/.n8n.

To jest ważne: w Twoim wariancie trwałość zapewnia tylko wolumen, nie kontener. Sama komenda docker run tworzy nowy egzemplarz aplikacji, ale podpina do niej ten sam magazyn danych.

Jak to wygląda w praktyce
Przykładowy przebieg:

uruchamiasz docker run ...,

Docker tworzy lub podłącza n8n_data,

startuje kontener n8n,

otwierasz localhost:5678,

zapisujesz workflow,

workflow trafia do /home/node/.n8n,

zatrzymujesz kontener,

uruchamiasz go ponownie,

dane wracają z wolumenu.

Co oznacza -d
Flaga -d uruchamia kontener w tle, więc terminal nie pozostaje „zajęty” przez proces n8n. To nie zmienia przepływu danych, tylko sposób pracy procesu na komputerze.

Istotne ograniczenie
W tej wersji nie ma jeszcze:

reverse proxy,

publicznego adresu URL,

osobnej bazy PostgreSQL,

uwierzytelniania na poziomie wejścia do panelu,

zabezpieczeń dla ekspozycji do internetu.

Czyli to jest dobry układ do lokalnego uruchomienia i testów na stanowisku, ale nie pełna produkcja wystawiona publicznie. n8n w dokumentacji wyraźnie podkreśla, że self-hosting wymaga wiedzy administracyjnej i ostrożności, bo błędy mogą prowadzić do utraty danych albo problemów bezpieczeństwa.

W jednej linijce
Przepływ jest taki: przeglądarka → port 5678 hosta → kontener n8n → zapis do wolumenu n8n_data → ponowne odczytanie po restarcie.